# Solar filament offline inference and submission CSV

Run this notebook inside Kaggle with the competition data and trained Kaggle Model attached through **Add Input**. Use a GPU for practical inference speed. The notebook reads only local attached files under `/kaggle/input`; Internet can remain disabled. It requires no Kaggle account credentials, download calls or API submission settings.

Run all cells to create the validated **`/kaggle/working/submission.csv`**. All project methods are included here, one function per cell. No external project `.py` files are needed, and no retraining occurs.

The notebook produces predictions; Kaggle's competition submission/scoring workflow evaluates them against hidden labels. Creating the CSV by itself does not produce a leaderboard score. After Save Version completes, use the competition's Submit Prediction workflow to select/upload the output file, as supported by the competition. This notebook does not access hidden labels or make network requests to obtain a score.

## Competition requirements and scoring scope

This notebook implements the supplied competition overview's prediction-file requirements:

- Predictions use only the provided test H-alpha images as model inputs. Attached trained weights and inference settings define the fixed model; no training annotations, chirality labels, spines, magnetograms or hidden labels are read during inference.
- Produce one CSV covering the complete test set, with exactly `filament_id` and `segmentation_rle` columns.
- Each row is one predicted filament. Preserve the original image identifier and append a unique instance suffix. The suffix does not need to match a ground-truth annotation index.
- Store only COCO compressed RLE counts in each CSV field, without manually wrapping counts in quotation marks or including RLE sizes. The CSV writer handles CSV formatting. Masks are restored to 2048 × 2048 pixels before encoding.
- The number of predictions need not match the hidden number of filaments. Images without detections have no prediction rows, and remain recorded in the coverage audit.

The updated leaderboard description specifies Panoptic Quality, although the supplied overview retains older Dice wording and inconsistent announcement thresholds. This notebook reports no local hidden-test score and makes no conversion from our local reference PQ to an official score. Kaggle's scorer determines the result after the CSV is submitted.

Running offline inside Kaggle is our chosen execution strategy, not a requirement established by the supplied overview. Attached model discovery, confidence thresholds and nonoverlapping pixel ownership are implementation choices, not additional competition rules. Offline execution requires the relevant dependencies and trained model to be attached/available before the run.

This workflow ends with the validated submission CSV and Kaggle's external scoring workflow. Final-report, prize and repository-release preparation are outside its scope.


## 1. Offline environment checks

Use Kaggle's installed Torch/Torchvision pair and helper packages. No runtime package installations or downloads occur. Missing packages cause an explicit setup error; select an environment containing them or attach the required dependency wheels and install those offline before running. The trained model bundle includes its original `requirements.txt` for reference; do not reinstall Torch blindly.

In [ ]:
import importlib.util

required = ['torch', 'torchvision', 'numpy', 'pandas', 'PIL', 'pycocotools', 'matplotlib', 'tqdm']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f'Missing offline dependencies: {missing}. Use the model requirements and locally attached wheels; no automatic installation is attempted.')

In [ ]:
from collections import defaultdict
import csv
import hashlib
import importlib.metadata
import json
import re
import time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from pycocotools import mask as coco_mask
from tqdm.auto import tqdm
import torch
import torch.nn.functional as F
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn

## 2. Local input configuration

Attach your trained model version and competition data before running or saving a version. Leave paths blank for automatic discovery of exactly one matching local input; set exact paths if multiple candidates are attached.

`model_directory` is the folder containing `model.pt`, `inference_config.json`, `training_config.json` and `manifest.json`. No model handle or version lookup is used: the attached files are the source of truth, recorded by their path and checkpoint hash.

Leave threshold overrides as `None` to use the model's saved inference settings. Prediction runs automatically. Diagnostics are saved separately; only the validated CSV is copied to Kaggle's standard output location.

In [ ]:
@dataclass
class Config:
    input_directory: str = '/kaggle/input'
    test_directory: str = ''  # Optional exact attached test_images folder.
    model_directory: str = ''  # Optional exact attached model bundle folder.
    output_directory: str = '/kaggle/working/solar-filament-inference'
    submission_path: str = '/kaggle/working/submission.csv'
    score_threshold_override: float | None = None
    mask_threshold_override: float | None = None
    preview_count: int = 3
    device: str = 'auto'

CFG = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if CFG.device == 'auto' else torch.device(CFG.device)
OUTPUT = Path(CFG.output_directory)
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE, '| Torch:', torch.__version__, '| Torchvision:', torchvision.__version__)

### Locate the attached model bundle

Read existing local files only. If multiple complete bundles are attached, require an explicit folder rather than silently selecting another model/version.

In [ ]:
def resolve_model_directory(cfg):
    required = ['model.pt', 'inference_config.json', 'training_config.json', 'manifest.json']
    if cfg.model_directory:
        candidates = [Path(cfg.model_directory)]
    else:
        root = Path(cfg.input_directory)
        if not root.is_dir():
            raise FileNotFoundError(f'Kaggle input directory not found: {root}')
        candidates = sorted({path.parent for path in root.rglob('inference_config.json')
                             if all((path.parent / name).is_file() for name in required)})
    if len(candidates) != 1:
        raise ValueError(f'Expected one attached model bundle, found {len(candidates)}: {candidates}. Attach the model or set model_directory explicitly.')
    folder = candidates[0]
    for name in required:
        if not (folder / name).is_file():
            raise FileNotFoundError(f'Missing model bundle file: {folder / name}')
    return folder

### Hash a file

Record model and submission hashes for reproducibility.

In [ ]:
def file_sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

### Read saved inference settings

Require the same normalization and overlap policy as the training notebook. Refuse a bundle that needs an unsupported processing path.

In [ ]:
def read_inference_settings(folder, cfg):
    settings = json.loads((folder / 'inference_config.json').read_text())
    training = json.loads((folder / 'training_config.json').read_text())
    expected = 'grayscale / 255, repeat 3 channels; internal model normalization'
    if settings.get('normalization') != expected:
        raise ValueError('Unsupported normalization; review the model bundle')
    if settings.get('overlap_policy') != 'confidence-ordered pixel ownership; no component filtering':
        raise ValueError('Unsupported mask overlap policy')
    assert settings['image_size'] == training['image_size']
    assert settings['max_detections'] == training['max_detections']
    if cfg.score_threshold_override is not None:
        settings['score_threshold'] = cfg.score_threshold_override
    if cfg.mask_threshold_override is not None:
        settings['mask_threshold'] = cfg.mask_threshold_override
    assert 0 <= settings['score_threshold'] <= 1
    assert 0 < settings['mask_threshold'] <= 1
    assert settings['image_size'] >= 64 and settings['max_detections'] >= 1
    return settings

### Reconstruct the trained model

No pretrained download is needed. Load the state dictionary strictly and use the saved resolution and detection cap. weights_only=True avoids unrestricted pickle loading.

In [ ]:
def load_model(folder, settings, device):
    checkpoint = torch.load(folder / 'model.pt', map_location='cpu', weights_only=True)
    assert {'model', 'config', 'epoch', 'fold'}.issubset(checkpoint)
    assert checkpoint['config']['image_size'] == settings['image_size']
    assert checkpoint['config']['max_detections'] == settings['max_detections']
    model = maskrcnn_resnet50_fpn(
        weights=None, weights_backbone=None, num_classes=2,
        min_size=settings['image_size'], max_size=settings['image_size'],
        box_score_thresh=0.0, box_detections_per_img=settings['max_detections'],
        trainable_backbone_layers=5)
    model.load_state_dict(checkpoint['model'], strict=True)
    manifest = json.loads((folder / 'manifest.json').read_text())
    print('Trained fold:', checkpoint['fold'], '| epoch:', checkpoint['epoch'] + 1)
    print('Training versions:', manifest.get('torch'), manifest.get('torchvision'))
    print('Model SHA256:', file_sha256(folder / 'model.pt'))
    return model.to(device).eval()

In [ ]:
MODEL_DIRECTORY = resolve_model_directory(CFG)
SETTINGS = read_inference_settings(MODEL_DIRECTORY, CFG)
MODEL = load_model(MODEL_DIRECTORY, SETTINGS, DEVICE)
print('Inference settings:', SETTINGS)

## 3. Test image audit and preprocessing

Use only the attached competition's test H-alpha images. Resolve a local test_images directory, then verify the complete image set. Filename stems become submission image identifiers. Never use training images as a fallback.

### Locate the attached competition test directory

Require a single test_images folder or an explicit configured path. Missing or ambiguous input stops before prediction.

In [ ]:
def resolve_test_directory(cfg):
    if cfg.test_directory:
        folder = Path(cfg.test_directory)
        if not folder.is_dir():
            raise FileNotFoundError(f'Test image directory not found: {folder}')
        return folder
    root = Path(cfg.input_directory)
    candidates = sorted(path for path in root.rglob('test_images') if path.is_dir())
    if len(candidates) != 1:
        raise ValueError(f'Expected one attached test_images folder, found {len(candidates)}: {candidates}. Attach competition data or set test_directory explicitly.')
    return candidates[0]

### Audit the full test set

Check every image is 2048 × 2048. Sort filenames to make output order deterministic.

In [ ]:
def audit_test_images(directory):
    directory = Path(directory)
    if not directory.is_dir():
        raise FileNotFoundError(f'Test image directory not found: {directory}')
    paths = sorted(path for path in directory.iterdir() if path.suffix.lower() in {'.jpg', '.jpeg'})
    if not paths or len({path.stem for path in paths}) != len(paths):
        raise ValueError('No test images or duplicate image identifiers')
    for path in paths:
        if not re.fullmatch(r'\d{14}[A-Za-z]{2}', path.stem):
            raise ValueError(f'Unexpected image identifier: {path.stem}')
        with Image.open(path) as image:
            assert image.size == (2048, 2048), f'Unexpected image size: {path.name}'
            image.verify()
    print('Test images:', len(paths))
    return paths

### Prepare one image

Repeat the grayscale brightness channel three times. Divide by 255 once; the model performs backbone normalization internally. Resize bilinearly to the saved input size.

In [ ]:
def prepare_image(path, size):
    with Image.open(path) as source:
        image = source.convert('L')
        shape = (image.height, image.width)
        resized = image.resize((size, size), Image.Resampling.BILINEAR)
        pixels = np.array(resized, dtype=np.float32) / 255.0
    return torch.from_numpy(pixels).unsqueeze(0).repeat(3, 1, 1), shape

In [ ]:
TEST_DIRECTORY = resolve_test_directory(CFG)
TEST_PATHS = audit_test_images(TEST_DIRECTORY)

## 4. Separate instance masks and COCO RLE

This reproduces training-time validation: confidence filtering, bilinear restoration of probabilities to original coordinates, thresholding, and confidence-ordered pixel ownership. A lower-confidence mask cannot reclaim pixels assigned to a higher-confidence mask. Empty masks are discarded, with no component splitting or size filter.

CSV quoting added by the CSV writer is formatting, not part of the encoded counts. Never manually add quotation marks around the RLE string.

### Encode one mask

Use Fortran-order arrays and COCO compressed counts, not another competition’s start-length encoding.

In [ ]:
def encode_mask(mask):
    encoded = coco_mask.encode(np.asfortranarray(mask.astype(np.uint8)))
    return {'size': list(encoded['size']), 'counts': encoded['counts'].decode('ascii')}

### Decode one RLE

The submission stores counts only; dimensions are fixed at 2048 × 2048.

In [ ]:
def decode_rle(counts, shape=(2048, 2048)):
    return coco_mask.decode({'size': list(shape), 'counts': counts.encode('ascii')})

### Restore and separate instances

Process one original-resolution probability mask at a time to limit peak memory. Ordering matches the training notebook.

In [ ]:
def postprocess_prediction(prediction, shape, settings):
    occupied = np.zeros(shape, dtype=bool)
    instances = []
    for index in prediction['scores'].argsort(descending=True).tolist():
        score = prediction['scores'][index].item()
        if score < settings['score_threshold']:
            continue
        probability = prediction['masks'][index:index + 1].float()
        restored = F.interpolate(probability, size=shape, mode='bilinear', align_corners=False)
        mask = (restored[0, 0].cpu().numpy() >= settings['mask_threshold']) & ~occupied
        if mask.any():
            instances.append({'rle': encode_mask(mask), 'score': score, 'area': int(mask.sum())})
            occupied |= mask
    return instances

### Predict one image

Inference mode disables gradients. No automatic mixed precision or augmentation is added.

In [ ]:
@torch.inference_mode()
def predict_image(model, path, settings, device):
    image, shape = prepare_image(path, settings['image_size'])
    prediction = model([image.to(device)])[0]
    return postprocess_prediction(prediction, shape, settings)

### Verify one encoded instance

Check nonempty binary content, dimensions and exact encode/decode round-trip stability.

In [ ]:
def verify_instance(encoded):
    assert encoded['size'] == [2048, 2048]
    mask = decode_rle(encoded['counts'])
    assert mask.shape == (2048, 2048) and mask.any()
    assert encode_mask(mask) == encoded, 'RLE round-trip failed'
    return int(mask.sum())

## 5. Generate the complete submission

Predict one full image at a time. Each nonempty instance becomes one row. Images with no detections have no CSV rows, but are explicitly recorded in the coverage file. Per-image RLE JSON files support later previews without rerunning the model.

The CSV is installed atomically only after inference finishes. Use a fresh output directory for each experiment; this prevents mixing old outputs with a new model. A failed run leaves diagnostic temporary files and should be restarted in a fresh output directory.

### Reserve a fresh prediction run

Refuse to mix existing diagnostic outputs with a new run. Choose a fresh output_directory for each experiment. Kaggle's final submission.csv is written only after complete validation.

In [ ]:
def reserve_run(output):
    markers = ['run_manifest.json', 'submission.csv', 'submission.csv.tmp', 'instances']
    if any((output / name).exists() for name in markers):
        raise FileExistsError('Prediction outputs already exist; choose a fresh output_directory')
    (output / 'instances').mkdir()

### Record model identity and settings

Record the exact attached checkpoint path, file hash, saved training manifest and effective thresholds. No remote version lookup is required.

In [ ]:
def write_run_manifest(folder, settings, cfg, output):
    manifest = {'model_directory': str(folder), 'model_sha256': file_sha256(folder / 'model.pt'),
                'settings': settings, 'config': asdict(cfg),
                'torch': torch.__version__, 'torchvision': torchvision.__version__,
                'source_manifest': json.loads((folder / 'manifest.json').read_text())}
    (output / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
    packages = ['torch', 'torchvision', 'numpy', 'pandas', 'Pillow', 'pycocotools']
    versions = [f'{name}=={importlib.metadata.version(name)}' for name in packages]
    (output / 'requirements.txt').write_text('\n'.join(versions) + '\n')

### Write predictions and coverage

Image identifiers remain unchanged. Instance suffixes are unique within each image. Per-image runtime includes preprocessing and encoding.

In [ ]:
def generate_submission(model, paths, settings, device, output):
    coverage, detections = [], []
    temporary = output / 'submission.csv.tmp'
    with open(temporary, 'w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=['filament_id', 'segmentation_rle'])
        writer.writeheader()
        for path in tqdm(paths, desc='Predicting test images'):
            started = time.perf_counter()
            instances = predict_image(model, path, settings, device)
            for index, instance in enumerate(instances, 1):
                area = verify_instance(instance['rle'])
                identity = f'{path.stem}_{index}'
                writer.writerow({'filament_id': identity, 'segmentation_rle': instance['rle']['counts']})
                detections.append({'filament_id': identity, 'score': instance['score'], 'area': area})
            (output / 'instances' / f'{path.stem}.json').write_text(json.dumps(instances))
            coverage.append({'filename': path.name, 'image_id': path.stem,
                             'predicted_instances': len(instances),
                             'seconds': time.perf_counter() - started})
            pd.DataFrame(coverage).to_csv(output / 'test_coverage.csv', index=False)
    destination = output / 'submission.csv'
    temporary.replace(destination)
    pd.DataFrame(detections, columns=['filament_id', 'score', 'area']).to_csv(output / 'detections.csv', index=False)
    return destination

In [ ]:
reserve_run(OUTPUT)
write_run_manifest(MODEL_DIRECTORY, SETTINGS, CFG, OUTPUT)
SUBMISSION_PATH = generate_submission(MODEL, TEST_PATHS, SETTINGS, DEVICE, OUTPUT)
print('Generated CSV awaiting validation:', SUBMISSION_PATH)

## 6. Validate the actual CSV file

Read the CSV back, not just in-memory predictions. Check its headers, identifiers, counts, RLE content, nonoverlapping masks and image coverage against the complete test directory. No prediction is required for an image with zero detections. An entirely empty submission is blocked for inspection.

Passing these checks proves structural consistency, not segmentation quality or an official score. Inspect overlays before submitting.

### Validate rows and decoded masks

Group by image while decoding to keep memory bounded. Detect duplicate instance IDs, unknown images and mask overlaps.

In [ ]:
def validate_submission(path, paths, output):
    expected = {p.stem for p in paths}
    rows = defaultdict(list)
    identities = set()
    with open(path, newline='') as stream:
        reader = csv.DictReader(stream)
        assert reader.fieldnames == ['filament_id', 'segmentation_rle'], 'Wrong CSV columns'
        for row in reader:
            identity, counts = row['filament_id'], row['segmentation_rle']
            assert identity not in identities, 'Duplicate filament ID'
            image_id, suffix = identity.rsplit('_', 1)
            assert image_id in expected and suffix.isdigit() and int(suffix) >= 1
            assert counts and not counts.startswith(('"', "'")), 'Malformed RLE counts'
            identities.add(identity)
            rows[image_id].append(counts)
    assert identities, 'No predicted instances: inspect model/settings before submitting'
    for image_id, counts_list in rows.items():
        occupied = np.zeros((2048, 2048), dtype=bool)
        for counts in counts_list:
            mask = decode_rle(counts)
            assert mask.shape == (2048, 2048) and mask.any()
            assert not np.any(occupied & (mask > 0)), f'Overlapping predictions: {image_id}'
            assert encode_mask(mask)['counts'] == counts
            occupied |= mask > 0
    coverage = pd.read_csv(output / 'test_coverage.csv')
    assert len(coverage) == len(paths) and coverage.image_id.is_unique
    assert set(coverage.image_id) == expected, 'Incomplete test coverage'
    for row in coverage.itertuples():
        assert row.predicted_instances == len(rows.get(row.image_id, [])), 'CSV/coverage mismatch'
    report = {'images_processed': len(paths), 'instances': len(identities),
              'zero_detection_images': int((coverage.predicted_instances == 0).sum()),
              'csv_sha256': file_sha256(path), 'status': 'structural validation passed'}
    (output / 'validation_report.json').write_text(json.dumps(report, indent=2))
    return report

In [ ]:
VALIDATION_REPORT = validate_submission(SUBMISSION_PATH, TEST_PATHS, OUTPUT)
print(json.dumps(VALIDATION_REPORT, indent=2))
COVERAGE = pd.read_csv(OUTPUT / 'test_coverage.csv')
print(COVERAGE.predicted_instances.describe())
print('Mean seconds per image:', COVERAGE.seconds.mean())

### Preview saved instance contours

Read cached predictions. Different colors distinguish predicted objects. Include a zero-detection image in your manual review if any exist.

In [ ]:
def preview_predictions(paths, output, count):
    for path in paths[:count]:
        instances = json.loads((output / 'instances' / f'{path.stem}.json').read_text())
        with Image.open(path) as source:
            gray = np.array(source.convert('L'))
        fig, axis = plt.subplots(figsize=(9, 9))
        axis.imshow(gray, cmap='gray')
        colors = plt.cm.tab20(np.linspace(0, 1, max(1, len(instances))))
        for instance, color in zip(instances, colors):
            axis.contour(decode_rle(instance['rle']['counts']), levels=[0.5], colors=[color], linewidths=0.8)
        axis.set_title(f'{path.name}: {len(instances)} predicted filaments')
        axis.axis('off')
        fig.savefig(output / f'preview_{path.stem}.png', bbox_inches='tight')
        plt.show()
        plt.close(fig)

In [ ]:
preview_predictions(TEST_PATHS, OUTPUT, CFG.preview_count)

## Files produced and scoring

The final cell writes **`/kaggle/working/submission.csv`** after structural validation. This is the file to submit through Kaggle's competition interface. Diagnostics remain under `/kaggle/working/solar-filament-inference`:

- `test_coverage.csv`: every test image, including zero-detection images.
- `detections.csv`: confidence and pixel area per retained instance.
- `run_manifest.json`, `requirements.txt`: attached model identity and effective settings.
- `validation_report.json`: structural checks and submission hash.
- `instances/`, `preview_*.png`: individual masks and visual review.

Save a notebook version with outputs. Kaggle scores submitted predictions against hidden labels outside this notebook; no local hidden-test score is available. This baseline applies no ensemble or test-time augmentation.

References: [Torchvision Mask R-CNN](https://docs.pytorch.org/vision/main/models/generated/torchvision.models.detection.maskrcnn_resnet50_fpn.html), [competition](https://www.kaggle.com/competitions/filament-segmentation-2026).

## 7. Write Kaggle's submission output

Revalidate immediately before copying. The final file is `/kaggle/working/submission.csv`; Kaggle can collect it as a notebook output. No download, login, upload or submission API is called. If an earlier cell fails, this final publication step does not run.

In [ ]:
def publish_submission(path, paths, cfg, output):
    import shutil
    report = validate_submission(path, paths, output)
    destination = Path(cfg.submission_path)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.resolve() != path.resolve():
        temporary = destination.with_suffix('.csv.tmp')
        shutil.copyfile(path, temporary)
        assert file_sha256(temporary) == report['csv_sha256'], 'Copy checksum mismatch'
        temporary.replace(destination)
    print('Submission CSV ready:', destination)
    print('Instances:', report['instances'], '| Images processed:', report['images_processed'])
    print('Use this file in Kaggle’s competition submission workflow to obtain the hidden-test score.')
    return destination

In [ ]:
KAGGLE_SUBMISSION_PATH = publish_submission(SUBMISSION_PATH, TEST_PATHS, CFG, OUTPUT)